In [1]:
import io
import pathlib

import numpy as np
import pandas as pd
from minio import Minio

MINIO_ENDPOINT_URL = "nonarithmetically-undeliberating-janelle.ngrok-free.app"
ACCESS_KEY = "slcomp"
SECRET_KEY = "slcomp@data"
client = Minio(
    MINIO_ENDPOINT_URL,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=True,
)

In [2]:
database_df_object = client.get_object("slcomp", "Data/Database.csv").data
database_df = pd.read_csv(
    io.StringIO(database_df_object.decode("utf-8")), low_memory=False, dtype=object
)
database_consolidated_df_object = client.get_object(
    "slcomp", "Data/Consolidated_Data.csv"
).data
database_consolidated_df = pd.read_csv(
    io.StringIO(database_consolidated_df_object.decode("utf-8")),
    low_memory=False,
    dtype=object,
)

In [3]:
dictionary = {}
dictionary["All"] = {"JNAME": database_df.JNAME.values}

for ref in np.unique(np.hstack(database_df.Reference.str.split(" § "))):
    jnames = database_df[
        database_df.Reference.str.contains(ref, regex=False)
    ].JNAME.values
    dictionary[ref] = {"JNAME": jnames}

np.save("dictionary.npy", dictionary)

In [4]:
def get_proper_values_scalar(item):
    """
    Processes a single item from a DataFrame cell.
    If the item contains multiple values separated by " § ", it splits them into a list.
    Otherwise, it returns the item as a string (or NaN if it's NaN).
    This is used to handle fields where multiple references or values
    for a single object are concatenated in the raw CSV.
    """
    if pd.isna(item):
        return item

    value = str(item)

    if " § " in value:
        return value.split(" § ")
    else:
        return value

In [5]:
for key in database_df.keys():
    database_df[key] = database_df[key].apply(get_proper_values_scalar)

In [6]:
# Renaming keys for print in Latex when displaying:
keys = {
    "Original_ID": r"Original ID",
    "Alternative_Name": r"Alternative Name",
    "Individual_Coordinates": r"Individual Coordinates",
    "z_L": r"$z_L$",
    "z_LErr": r"$\delta z_L$",
    "z_LType": r"$z_L^{\mathrm{Type}}$",
    "z_LRef": r"$z_L^{\mathrm{Ref}}$",
    "z_S": r"$z_S$",
    "z_SErr": r"$\delta z_S$",
    "z_SType": r"$z_S^{\mathrm{Type}}$",
    "z_SRef": r"$z_S^{\mathrm{Ref}}$",
    "velDisp": r"$\sigma_v$",
    "velDispErr": r"$\delta\sigma_v$",
    "velDispRef": r"$\sigma_v^{\mathrm{Ref}}$",
    "theta_E": r"$\theta_E$",
    "theta_EErr": r"$\delta\theta_E$",
    "theta_EMethod": r"$\theta_E^{\mathrm{Method}}$",
    "theta_ERef": r"$\theta_E^{\mathrm{Ref}}$",
    "mag_u": r"$m_u$",
    "mag_uErr": r"$\delta m_u$",
    "mag_uS": r"$m_u^{\mathrm{Source}}$",
    "mag_uRef": r"$m_u^{\mathrm{Ref}}$",
    "mag_uSRef": r"$m_u^{\mathrm{Source\,Ref}}$",
    "mag_g": r"$m_g$",
    "mag_gErr": r"$\delta m_g$",
    "mag_gS": r"$m_g^{\mathrm{Source}}$",
    "mag_gRef": r"$m_g^{\mathrm{Ref}}$",
    "mag_gSRef": r"$m_g^{\mathrm{Source\,Ref}}$",
    "mag_r": r"$m_r$",
    "mag_rErr": r"$\delta m_r$",
    "mag_rS": r"$m_r^{\mathrm{Source}}$",
    "mag_rRef": r"$m_r^{\mathrm{Ref}}$",
    "mag_rSRef": r"$m_r^{\mathrm{Source\,Ref}}$",
    "mag_i": r"$m_i$",
    "mag_iErr": r"$\delta m_i$",
    "mag_iS": r"$m_i^{\mathrm{Source}}$",
    "mag_iRef": r"$m_i^{\mathrm{Ref}}$",
    "mag_iSRef": r"$m_i^{\mathrm{Source\,Ref}}$",
    "mag_z": r"$m_z$",
    "mag_zErr": r"$\delta m_z$",
    "mag_zS": r"$m_z^{\mathrm{Source}}$",
    "mag_zRef": r"$m_z^{\mathrm{Ref}}$",
    "mag_zSRef": r"$m_z^{\mathrm{Source\,Ref}}$",
    "mag_y": r"$m_y$",
    "mag_yErr": r"$\delta m_y$",
    "mag_yS": r"$m_y^{\mathrm{Source}}$",
    "mag_yRef": r"$m_y^{\mathrm{Ref}}$",
    "mag_ySRef": r"$m_y^{\mathrm{Source\,Ref}}$",
    "mag_F814W": r"$m_{F814W}$",
    "mag_F814WErr": r"$\delta m_{F814W}$",
    "mag_F814WS": r"$m_{F814W}^{\mathrm{Source}}$",
    "mag_F814WRef": r"$m_{F814W}^{\mathrm{Ref}}$",
    "mag_F814WSRef": r"$m_{F814W}^{\mathrm{Source\,Ref}}$",
    "System_Type": r"System Type",
    "Lens_Type": r"Lens Type",
    "Source_Type": r"Source Type",
    "Other_Matches": r"Other Matches",
    "Original_Comments": r"Original Comments",
}

In [7]:
database_df = database_df.rename(columns=keys)
database_df.to_pickle("database_df.pickle")

database_consolidated_df = database_consolidated_df.rename(columns=keys)
database_consolidated_df.to_parquet("database_consolidated_df.parquet")

In [8]:
cutouts_df_object = client.get_object(
    "slcomp", "Cutouts/Processed_Cutouts.parquet"
).data
cutouts_df = pd.read_parquet(io.BytesIO(cutouts_df_object))
cutouts_df = cutouts_df.query('cutout_size=="20asec"').reset_index(drop=True)
cutouts_df["file_strip"] = cutouts_df.file_name.apply(lambda x: pathlib.Path(x).stem)

In [9]:
cutouts_fits_object = client.get_object("slcomp", "Cutouts/FITS.parquet").data
cutouts_fits = pd.read_parquet(io.BytesIO(cutouts_fits_object))
cutouts_fits["file_strip"] = cutouts_fits.file_name.apply(
    lambda x: pathlib.Path(x).stem
)
cutouts_fits["file_strip"] = cutouts_fits["file_strip"].str.split(".fits").str[0]

In [10]:
cutouts = pd.merge(
    cutouts_fits,
    cutouts_df,
    on=["file_strip", "JNAME", "survey", "cutout_size"],
    how="outer",
)

In [11]:
cutouts.keys()

Index(['JNAME', 'survey', 'cutout_size', 'band', 'tile', 'file_name_x',
       'file_path_x', 'file_strip', 'processing', 'is_rgb', 'file_name_y',
       'file_path_y'],
      dtype='object')

In [12]:
cutouts = cutouts[
    [
        "JNAME",
        "survey",
        "cutout_size",
        "band",
        "tile",
        "is_rgb",
        "processing",
        "file_name_y",
        "file_path_y",
    ]
]
cutouts = cutouts.rename(
    columns={"file_path_y": "file_path", "file_name_y": "file_name"}
)

In [13]:
cutouts = cutouts[cutouts.file_path.notnull()].reset_index(drop=True)

In [14]:
idxs = cutouts[~cutouts.band.notnull()].index
cutouts.loc[idxs, "band"] = cutouts.loc[idxs, "processing"]

In [15]:
cutouts.band = pd.Categorical(
    cutouts.band, categories=["u", "g", "r", "i", "z", "y", "trilogy", "lsb"]
)

In [16]:
cutouts.to_parquet("cutouts.parquet")